# Lab 23: Prompt Injection and Layered Controls

*AI Cybersecurity Masterclass · Lab 23 · AI60secLab*

**Companion video:** [▶ How Prompt Injection Works in Cybersecurity in 60 Seconds](https://www.youtube.com/watch?v=ZEZMfywIPp0&list=PLG4l6Ssi9A00)

**Research question:** When untrusted text starts giving an AI assistant instructions, which controls actually stop the attack, and what does each one cost?

A SOC assistant summarizes threat intelligence, drafts tickets, and can email the SOC. Attackers plant instructions directly in a user message or indirectly inside a retrieved document. We measure attack success and legitimate-task success under each control layer.

**Learning objectives**
- separate direct from indirect prompt injection;
- explain why trusted instructions and untrusted data must stay separate;
- measure attack success rate (ASR) and benign task success for each control stack;
- show that the model proposes and deterministic code decides;
- turn blocked actions and flagged inputs into measurable log events.

> **About the simulated assistant.** These labs use `SimulatedModel`, a deterministic stand-in for an LLM. It reproduces a documented failure of real models: it understands instructions written in many phrasings, including ones hidden in retrieved data, and proposes actions based on them. That keeps every result repeatable and free to run. The *controls* you build and measure are real code that could sit in front of a real model unchanged. What the simulation cannot tell you is how often a specific production model falls for a specific payload: that needs testing against that model.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
import matplotlib.pyplot as plt

from src.llm_app import (
    SYSTEM_PROMPT,
    ATTACK_PAYLOADS,
    ControlConfig,
    build_test_cases,
    evaluate_controls,
    run_pipeline,
)

cases = build_test_cases()
summary = pd.DataFrame(cases)[["kind", "family", "channel"]].value_counts().reset_index(name="cases")
display(summary)
print("System prompt (contains a planted secret we must never leak):")
print(SYSTEM_PROMPT)

## Step 1: The attack payloads

Six ways to say the same thing. A human reader recognizes all of them as instructions. So does an LLM. A keyword filter recognizes only the ones it was written for.

In [ ]:
display(pd.Series(ATTACK_PAYLOADS, name="payload").to_frame())

## Step 2: No controls

The undefended assistant acts on any instruction it reads, wherever it came from. Notice the provenance: without trust separation, the attacker's instruction from an external blog *looks* exactly as trusted as the system prompt.

In [ ]:
example = next(c for c in cases if c["case_id"] == "indirect-paraphrased_override")
result = run_pipeline(example["user_request"], example["retrieved"], ControlConfig())
print("Answer:", result["answer"])
print("Executed tool calls:")
for call in result["executed"]:
    print(" ", call.tool, call.args, "| triggered by", call.provenance, "content")

## Step 3: Add controls one layer at a time

| Layer | What it does |
|---|---|
| input pattern filter | drops inputs that match known injection phrases |
| trust separation | marks retrieved content as untrusted data, so any action it triggers carries untrusted provenance |
| schema validation | tool calls must name a known tool with exactly the declared arguments |
| deterministic policy | the decision: least privilege, no external action from untrusted provenance, allowlisted recipients, no secrets in arguments |
| output secret filter | redacts the secret if it appears in the answer |

In [ ]:
stacks = {
    "0 none": ControlConfig(),
    "1 pattern filter": ControlConfig(input_pattern_filter=True),
    "2 schema validation": ControlConfig(schema_validation=True),
    "3 trust separation + policy": ControlConfig(trust_separation=True, deterministic_policy=True),
    "4 all input/action layers": ControlConfig(True, True, True, True, False),
    "5 all layers + output filter": ControlConfig(True, True, True, True, True),
}

rows, per_case, logs = [], [], {}
for name, controls in stacks.items():
    results, log = evaluate_controls(cases, controls)
    logs[name] = log
    attacks = results[results["kind"] == "attack"]
    benign = results[results["kind"] == "benign"]
    rows.append({
        "stack": name,
        "attack_success_rate": attacks["attack_succeeded"].mean(),
        "benign_task_success": benign["benign_completed"].mean(),
        "log_events": len(log.events),
    })
    per_case.append(results.assign(stack=name))

stack_results = pd.DataFrame(rows).set_index("stack")
display(stack_results.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
stack_results[["attack_success_rate", "benign_task_success"]].plot(kind="barh", ax=ax)
ax.set_xlim(0, 1.05)
ax.set_xlabel("Share of cases")
ax.set_title("Attack success falls only when layers combine; legitimate work keeps running")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Step 4: Which payloads beat which layer?

In [ ]:
all_cases = pd.concat(per_case)
matrix = (
    all_cases[all_cases["kind"] == "attack"]
    .pivot_table(index=["family", "channel"], columns="stack", values="attack_succeeded", aggfunc="max")
    .astype(int)
)
display(matrix)

Read the matrix column by column. The pattern filter only stops the payload that uses its exact words. The policy layer stops every *action*, but not secrets written into the answer text. Only the output filter catches those. No single layer is enough.

## Step 5: Make abuse measurable

Every flagged input, schema failure, blocked action, and redaction is a log event. Counted over time, these become a security signal instead of silent failures.

In [ ]:
final_log = logs["5 all layers + output filter"].frame()
display(final_log["event"].value_counts().rename("events").to_frame())
display(final_log.head(10))

## Interpretation

1. Why does the pattern filter fail against the paraphrased and authority-impersonation payloads?
2. Trust separation alone changes nothing. Why does it become powerful only when combined with the policy?
3. The direct injection comes from the user, so its provenance is trusted. Which rule still stops it?
4. Benign task success stayed at 1.0. Why is that measurement as important as attack success rate?
5. Which log event would you alert on first in production, and why?

**Takeaway:** no single prompt or filter solves LLM security. The model proposes, deterministic controls decide, and every block is logged so abuse becomes measurable.